In [ ]:
"""Train TGAN by distilling soft prototype assignments from reference PCA space."""
import logging
import math
import os
import random
import time
from collections import defaultdict

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

import graph, importlib
importlib.reload(graph)
from graph import NeighborFinder
import utils
importlib.reload(utils)
from utils import EarlyStopMonitor, RandEdgeSampler


# =========================
# Directly define all configs here
# =========================
DATA = "p0.8_mu0.2_1"

BATCH_SIZE = 256
NUM_NEIGHBORS = 20
NUM_EPOCH = 150

NUM_HEADS = 2
DROP_OUT = 0.1
GPU = 0
UNIFORM = False
USE_TIME = "time"
AGG_METHOD = "attn"
ATTN_MODE = "prod"
NUM_LAYER = 2
LEARNING_RATE = 1e-3

# TGAN internals
NODE_DIM = 128
TIME_DIM = 50

# reference-space random walk
TIME_WINDOW = 300.0
RW_NUM_STEPS = 5
SELF_LOOP_EPS = 1e-8

# prototype distillation
NUM_CLUSTERS = 5
PROTO_TEMPERATURE = 0.01   # 越小越接近硬标签，越大越平滑

# optional losses
LAMBDA_PROTO = 1.0
LAMBDA_LINK = 0.01
LAMBDA_DEC = 0.1

DEC_ALPHA = 1.0

SEED = 2020
MODEL_SAVE_PATH = f"./saved_models/tgan_proto_{DATA}.pth"
CHECKPOINT_PATH = f"./saved_checkpoints/tgan_proto_{DATA}.pth"
REF_PCA_CACHE_PATH = f"./processed/{DATA}_ref_pca_for_proto.npz"


# =========================
# Reproducibility
# =========================
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


# =========================
# Logger
# =========================
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger()
logger.setLevel(logging.DEBUG)
fh = logging.FileHandler(f"log/{time.time()}.log")
fh.setLevel(logging.DEBUG)
ch = logging.StreamHandler()
ch.setLevel(logging.WARN)
formatter = logging.Formatter("%(asctime)s - %(name)s - %(levelname)s - %(message)s")
fh.setFormatter(formatter)
ch.setFormatter(formatter)
logger.addHandler(fh)
logger.addHandler(ch)




# =========================
# Helper functions
# =========================
def get_device(gpu_idx=0):
    if torch.cuda.is_available():
        return torch.device(f"cuda:{gpu_idx}")
    if torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")


def build_window_adjacency(src_all, dst_all, ts_all, center_time, time_window):
    mask = np.abs(ts_all - center_time) <= time_window
    src_cut = src_all[mask]
    dst_cut = dst_all[mask]

    adj = defaultdict(dict)
    for u, v in zip(src_cut, dst_cut):
        if u == v:
            continue
        adj[u][v] = adj[u].get(v, 0.0) + 1.0
        adj[v][u] = adj[v].get(u, 0.0) + 1.0
    return adj


def k_hop_rw_distribution(start_node, adj, num_steps=3, self_loop_eps=1e-8):
    if start_node not in adj or len(adj[start_node]) == 0:
        return {int(start_node): 1.0}

    current = {int(start_node): 1.0}
    accum = defaultdict(float)

    for _ in range(num_steps):
        nxt = defaultdict(float)
        for u, mass in current.items():
            nbrs = adj.get(u, {})
            total_w = sum(nbrs.values()) + self_loop_eps
            if total_w <= self_loop_eps:
                nxt[u] += mass
                continue
            for v, w in nbrs.items():
                nxt[v] += mass * (w / total_w)
        current = nxt
        for node, prob in current.items():
            accum[node] += prob

    total = sum(accum.values())
    if total <= 0:
        return {int(start_node): 1.0}
    return {node: prob / total for node, prob in accum.items()}


def dist_dict_to_vec(dist_dict, num_nodes):
    vec = np.zeros(num_nodes, dtype=np.float32)
    for node, prob in dist_dict.items():
        vec[int(node)] = prob
    s = vec.sum()
    if s > 0:
        vec /= s
    return vec


def build_reference_vector_for_one(node, t, src_all, dst_all, ts_all, time_window, rw_num_steps, num_nodes):
    adj = build_window_adjacency(src_all, dst_all, ts_all, t, time_window)
    rw_dist = k_hop_rw_distribution(
        start_node=int(node),
        adj=adj,
        num_steps=rw_num_steps,
        self_loop_eps=SELF_LOOP_EPS,
    )
    return dist_dict_to_vec(rw_dist, num_nodes)


def build_all_unique_node_event_pairs(src_l, dst_l, ts_l):
    all_node_idx = np.concatenate([src_l, dst_l])
    all_event_time = np.concatenate([ts_l, ts_l])
    pairs = np.column_stack([all_node_idx, all_event_time])
    pairs = np.unique(pairs, axis=0)
    return pairs


def pair_key(node, t):
    return (int(node), float(t))


def precompute_reference_pca_and_soft_targets(
    src_l,
    dst_l,
    ts_l,
    time_window,
    rw_num_steps,
    num_nodes,
    pca_dim,
    num_clusters,
    temperature,
    cache_path=None,
):
    """
    返回：
      pair_to_ref_pca: dict[(node,t)] -> PCA向量
      pair_to_soft_q: dict[(node,t)] -> soft prototype assignment
      ref_centers_pca: PCA空间中的reference prototypes
    """
    if cache_path is not None and os.path.exists(cache_path):
        cache = np.load(cache_path, allow_pickle=True)
        pair_arr = cache["pairs"]
        ref_pca_mat = cache["ref_pca_mat"]
        soft_q_mat = cache["soft_q_mat"]
        ref_centers_pca = cache["ref_centers_pca"]

        pair_to_ref_pca = {
            pair_key(pair_arr[i, 0], pair_arr[i, 1]): ref_pca_mat[i]
            for i in range(len(pair_arr))
        }
        pair_to_soft_q = {
            pair_key(pair_arr[i, 0], pair_arr[i, 1]): soft_q_mat[i]
            for i in range(len(pair_arr))
        }
        logger.info(f"Loaded cached prototype targets from {cache_path}")
        return pair_to_ref_pca, pair_to_soft_q, ref_centers_pca

    pairs = build_all_unique_node_event_pairs(src_l, dst_l, ts_l)
    logger.info(f"Precomputing reference vectors for {len(pairs)} unique node-events...")

    ref_mat = np.zeros((len(pairs), num_nodes), dtype=np.float32)
    for i, (node, t) in enumerate(pairs):
        if i % 500 == 0:
            logger.info(f"reference vector progress: {i}/{len(pairs)}")
        ref_mat[i] = build_reference_vector_for_one(
            node=node,
            t=t,
            src_all=src_l,
            dst_all=dst_l,
            ts_all=ts_l,
            time_window=time_window,
            rw_num_steps=rw_num_steps,
            num_nodes=num_nodes,
        )

    pca_dim = min(pca_dim, ref_mat.shape[0], ref_mat.shape[1])
    logger.info(f"Fitting PCA on reference matrix with dim {pca_dim} ...")
    pca = PCA(n_components=pca_dim, random_state=SEED)
    ref_pca_mat = pca.fit_transform(ref_mat).astype(np.float32)

    logger.info("Running KMeans in reference PCA space ...")
    km = KMeans(
        n_clusters=num_clusters,
        random_state=SEED,
        n_init=20,
    )
    hard_labels = km.fit_predict(ref_pca_mat)
    ref_centers_pca = km.cluster_centers_.astype(np.float32)

    # soft prototype assignment in reference PCA space
    # q_ref(i,k) \propto exp(-||r_i-c_k||^2 / tau)
    d2 = ((ref_pca_mat[:, None, :] - ref_centers_pca[None, :, :]) ** 2).sum(axis=2)
    logits = -d2 / max(temperature, 1e-12)
    logits = logits - logits.max(axis=1, keepdims=True)
    soft_q_mat = np.exp(logits)
    soft_q_mat /= soft_q_mat.sum(axis=1, keepdims=True)

    pair_to_ref_pca = {
        pair_key(pairs[i, 0], pairs[i, 1]): ref_pca_mat[i]
        for i in range(len(pairs))
    }
    pair_to_soft_q = {
        pair_key(pairs[i, 0], pairs[i, 1]): soft_q_mat[i].astype(np.float32)
        for i in range(len(pairs))
    }

    if cache_path is not None:
        os.makedirs(os.path.dirname(cache_path), exist_ok=True)
        np.savez_compressed(
            cache_path,
            pairs=pairs,
            ref_pca_mat=ref_pca_mat,
            soft_q_mat=soft_q_mat.astype(np.float32),
            ref_centers_pca=ref_centers_pca,
            hard_labels=hard_labels.astype(np.int32),
        )
        logger.info(f"Saved prototype targets to {cache_path}")

    return pair_to_ref_pca, pair_to_soft_q, ref_centers_pca


def lookup_soft_targets(node_list, time_list, pair_to_soft_q, num_clusters):
    out = np.zeros((len(node_list), num_clusters), dtype=np.float32)
    for i, (node, t) in enumerate(zip(node_list, time_list)):
        out[i] = pair_to_soft_q[pair_key(node, t)]
    return torch.tensor(out, dtype=torch.float32)


def soft_assign_dec(z, centers, alpha=1.0):
    dist_sq = torch.sum((z.unsqueeze(1) - centers.unsqueeze(0)) ** 2, dim=2)
    q = 1.0 / (1.0 + dist_sq / alpha)
    q = q ** ((alpha + 1.0) / 2.0)
    q = q / (torch.sum(q, dim=1, keepdim=True) + 1e-12)
    return q


def target_distribution(q):
    weight = q ** 2 / (torch.sum(q, dim=0, keepdim=True) + 1e-12)
    p = weight / (torch.sum(weight, dim=1, keepdim=True) + 1e-12)
    return p


def dec_kl_loss(z, centers, alpha=1.0):
    q = soft_assign_dec(z, centers, alpha=alpha)
    p = target_distribution(q).detach()
    return F.kl_div(torch.log(q + 1e-12), p, reduction="batchmean")


def proto_distill_loss(logits, soft_targets):
    pred = torch.softmax(logits, dim=1)
    return F.kl_div(torch.log(pred + 1e-12), soft_targets, reduction="batchmean")


# =========================
# Load data
# =========================
g_df = pd.read_csv(f"./preprocess/{DATA}.csv")

src_l = g_df.source.values
dst_l = g_df.destination.values
e_idx_l = g_df.idx.values
ts_l = g_df.timestamp.values

max_idx = max(src_l.max(), dst_l.max())
num_nodes = max_idx + 1
num_edges = len(g_df)
num_instance = len(src_l)
idx_list = np.arange(num_instance)

n_feat_path = f"./processed/{DATA}_node.npy"
if os.path.exists(n_feat_path):
    n_feat = np.load(n_feat_path)
else:
    n_feat = np.eye(num_nodes, dtype=np.float32)

e_feat_path = f"./processed/{DATA}.npy"
if os.path.exists(e_feat_path):
    e_feat = np.load(e_feat_path)
else:
    e_feat = np.zeros((num_edges, n_feat.shape[1]), dtype=np.float32)

logger.info(f"num of training instances: {num_instance}")
logger.info(f"num of total nodes: {len(np.unique(np.concatenate([src_l, dst_l])))}")
logger.info(f"n_feat shape: {n_feat.shape}")
logger.info(f"e_feat shape: {e_feat.shape}")


# ============
# read GT 
GT_CSV = f"./syn_data/{DATA}.csv"

gt_df = pd.read_csv(GT_CSV)
gt_part = build_partition_dict_from_dataframe(
    gt_df,
    source_col="source",
    destination_col="destination",
    timestamp_col="timestamp",
    source_commu_col="source_commu",
    destination_commu_col="destination_commu",
)

logger.info(f"Loaded ground-truth partition with {len(gt_part)} node-time assignments")


# =========================
# Build graph
# =========================
full_adj_list = [[] for _ in range(max_idx + 1)]
for src, dst, eidx, ts in zip(src_l, dst_l, e_idx_l, ts_l):
    full_adj_list[src].append((dst, eidx, ts))
    full_adj_list[dst].append((src, eidx, ts))

full_ngh_finder = NeighborFinder(full_adj_list, uniform=UNIFORM)
train_rand_sampler = RandEdgeSampler(src_l, dst_l)


# =========================
# Model
# =========================
device = get_device(GPU)

tgan = TGAN(
    full_ngh_finder,
    n_feat,
    e_feat,
    num_layers=NUM_LAYER,
    use_time=USE_TIME,
    agg_method=AGG_METHOD,
    attn_mode=ATTN_MODE,
    seq_len=NUM_NEIGHBORS,
    n_head=NUM_HEADS,
    drop_out=DROP_OUT,
    node_dim=NODE_DIM,
    time_dim=TIME_DIM,
).to(device)

with torch.no_grad():
    probe_size = min(4, len(src_l))
    probe_emb = tgan.get_node_event_embeddings(
        src_l[:probe_size],
        ts_l[:probe_size],
        num_neighbors=NUM_NEIGHBORS
    )
    EMBED_DIM = probe_emb.shape[1]

logger.info(f"Actual TGAN embedding dim: {EMBED_DIM}")

# cluster head: embedding -> soft prototype assignment
cluster_head = nn.Linear(EMBED_DIM, NUM_CLUSTERS).to(device)

# optional DEC centers directly in raw embedding space
cluster_centers = nn.Parameter(torch.randn(NUM_CLUSTERS, EMBED_DIM, device=device))

optimizer = torch.optim.AdamW(
    list(tgan.parameters())
    + list(cluster_head.parameters())
    + [cluster_centers],
    lr=LEARNING_RATE
)

bce_criterion = nn.BCELoss()

logger.info("TGAN model created")
logger.info(f"cluster_head: {cluster_head}")
logger.info(f"cluster_centers shape: {tuple(cluster_centers.shape)}")


# =========================
# Precompute reference PCA + soft prototype targets
# =========================
PCA_DIM = min(NODE_DIM, num_nodes, len(np.unique(np.concatenate([src_l, dst_l]))))
pair_to_ref_pca, pair_to_soft_q, ref_centers_pca = precompute_reference_pca_and_soft_targets(
    src_l=src_l,
    dst_l=dst_l,
    ts_l=ts_l,
    time_window=TIME_WINDOW,
    rw_num_steps=RW_NUM_STEPS,
    num_nodes=num_nodes,
    pca_dim=PCA_DIM,
    num_clusters=NUM_CLUSTERS,
    temperature=PROTO_TEMPERATURE,
    cache_path=REF_PCA_CACHE_PATH,
)


# =========================
# Training loop
# =========================
num_batch = math.ceil(num_instance / BATCH_SIZE)
early_stopper = EarlyStopMonitor(max_round=100, higher_better=False)

for epoch in range(NUM_EPOCH):
    np.random.shuffle(idx_list)
    tgan.train()
    tgan.ngh_finder = full_ngh_finder

    epoch_proto_loss = []
    epoch_link_loss = []
    epoch_dec_loss = []
    epoch_total_loss = []

    logger.info(f"start epoch {epoch}")

    for k in range(num_batch):
        s_idx = k * BATCH_SIZE
        e_idx = min(num_instance, s_idx + BATCH_SIZE)
        batch_idx = idx_list[s_idx:e_idx]

        src_l_cut = src_l[batch_idx]
        dst_l_cut = dst_l[batch_idx]
        ts_l_cut = ts_l[batch_idx]

        size = len(src_l_cut)
        if size == 0:
            continue

        optimizer.zero_grad()

        # ===== 1) optional link prediction loss =====
        if LAMBDA_LINK > 0.0:
            _, dst_l_fake = train_rand_sampler.sample(size)
            pos_prob, neg_prob = tgan.contrast(
                src_l_cut, dst_l_cut, dst_l_fake, ts_l_cut, NUM_NEIGHBORS
            )
            pos_label = torch.ones(size, dtype=torch.float32, device=device)
            neg_label = torch.zeros(size, dtype=torch.float32, device=device)
            loss_link = bce_criterion(pos_prob, pos_label) + bce_criterion(neg_prob, neg_label)
        else:
            loss_link = torch.tensor(0.0, device=device)

        # ===== 2) raw TGAN embeddings =====
        src_emb = tgan.get_node_event_embeddings(
            src_l_cut, ts_l_cut, num_neighbors=NUM_NEIGHBORS
        )
        dst_emb = tgan.get_node_event_embeddings(
            dst_l_cut, ts_l_cut, num_neighbors=NUM_NEIGHBORS
        )

        # ===== 3) reference soft prototype targets =====
        src_soft_targets = lookup_soft_targets(
            node_list=src_l_cut,
            time_list=ts_l_cut,
            pair_to_soft_q=pair_to_soft_q,
            num_clusters=NUM_CLUSTERS,
        ).to(device)

        dst_soft_targets = lookup_soft_targets(
            node_list=dst_l_cut,
            time_list=ts_l_cut,
            pair_to_soft_q=pair_to_soft_q,
            num_clusters=NUM_CLUSTERS,
        ).to(device)

        # ===== 4) predict soft assignments from TGAN embedding =====
        src_logits = cluster_head(src_emb)
        dst_logits = cluster_head(dst_emb)

        loss_proto_src = proto_distill_loss(src_logits, src_soft_targets)
        loss_proto_dst = proto_distill_loss(dst_logits, dst_soft_targets)
        loss_proto = loss_proto_src + loss_proto_dst

        # ===== 5) optional DEC on raw embedding =====
        if LAMBDA_DEC > 0.0:
            batch_emb = torch.cat([src_emb, dst_emb], dim=0)
            loss_dec = dec_kl_loss(batch_emb, cluster_centers, alpha=DEC_ALPHA)
        else:
            loss_dec = torch.tensor(0.0, device=device)

        # ===== total =====
        loss = (
            LAMBDA_PROTO * loss_proto
            + LAMBDA_LINK * loss_link
            + LAMBDA_DEC * loss_dec
        )

        loss.backward()
        optimizer.step()

        epoch_proto_loss.append(loss_proto.item())
        epoch_link_loss.append(loss_link.item())
        epoch_dec_loss.append(loss_dec.item())
        epoch_total_loss.append(loss.item())

    mean_proto = float(np.mean(epoch_proto_loss)) if epoch_proto_loss else 0.0
    mean_link = float(np.mean(epoch_link_loss)) if epoch_link_loss else 0.0
    mean_dec = float(np.mean(epoch_dec_loss)) if epoch_dec_loss else 0.0
    mean_total = float(np.mean(epoch_total_loss)) if epoch_total_loss else 0.0

    # =========================
    # Evaluation every epoch
    # =========================
    tgan.eval()
    cluster_head.eval()

    # mode 1: cluster_head argmax
    pred_mode1 = infer_mode1_partition(
        tgan=tgan,
        cluster_head=cluster_head,
        src_arr=src_l,
        dst_arr=dst_l,
        ts_arr=ts_l,
        num_neighbors=NUM_NEIGHBORS,
        batch_size=512,
        device=device,
    )
    ami_mode1, ari_mode1 = dynamic_ami_ari(gt_part, pred_mode1)

    logger.info(
        "epoch {} | total {:.6f} | proto {:.6f} | link {:.6f} | dec {:.6f} "
        "| mode1 AMI {:.6f} ARI {:.6f} ".format(
            epoch,
            mean_total,
            mean_proto,
            mean_link,
            mean_dec,
            ami_mode1,
            ari_mode1,
        )
    )

    torch.save(
        {
            "epoch": epoch,
            "tgan_state_dict": tgan.state_dict(),
            "cluster_head_state_dict": cluster_head.state_dict(),
            "cluster_centers": cluster_centers.detach().cpu(),
            "optimizer_state_dict": optimizer.state_dict(),
            "loss": mean_total,
            "ref_centers_pca": ref_centers_pca,
        },
        CHECKPOINT_PATH,
    )

    if early_stopper.early_stop_check(mean_total):
        logger.info(f"Early stop triggered at epoch {epoch}")
        break


# =========================
# Save final model
# =========================
logger.info("Saving final model")
torch.save(
    {
        "tgan_state_dict": tgan.state_dict(),
        "cluster_head_state_dict": cluster_head.state_dict(),
        "cluster_centers": cluster_centers.detach().cpu(),
        "ref_centers_pca": ref_centers_pca,
    },
    MODEL_SAVE_PATH,
)
logger.info(f"Model saved to {MODEL_SAVE_PATH}")